# 01 · Data Loading & Inspection

Load the raw EIA hourly load data for PJM and run basic quality checks before any transformations.

In [17]:
import pandas as pd

## 1.1 Load raw CSV

Parse `timestamp` as a datetime column so time-based operations work correctly downstream.

In [18]:
df = pd.read_csv(
    "../data/raw/eia_hourly_load.csv",
    parse_dates=["timestamp"]
)
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
df.head()

Loaded 5,000 rows × 11 columns


,timestamp,respondent,respondent-name,type,type-name,load_mw,value-units,hour,dayofweek,month,year
0,2023-01-01 00:00:00,PJM,"PJM Interconnection, LLC",D,Demand,88145,megawatthours,0,6,1,2023
1,2023-01-01 00:00:00,PJM,"PJM Interconnection, LLC",DF,Day-ahead demand forecast,90038,megawatthours,0,6,1,2023
2,2023-01-01 00:00:00,PJM,"PJM Interconnection, LLC",NG,Net generation,92327,megawatthours,0,6,1,2023
3,2023-01-01 00:00:00,PJM,"PJM Interconnection, LLC",TI,Total interchange,4181,megawatthours,0,6,1,2023
4,2023-01-01 01:00:00,PJM,"PJM Interconnection, LLC",D,Demand,85805,megawatthours,1,6,1,2023


## 1.2 Schema inspection

Check dtypes and non-null counts. Expect 11 columns, `timestamp` as `datetime64`, numeric columns as `int64`.

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   timestamp        5000 non-null   datetime64[us]
 1   respondent       5000 non-null   str           
 2   respondent-name  5000 non-null   str           
 3   type             5000 non-null   str           
 4   type-name        5000 non-null   str           
 5   load_mw          5000 non-null   int64         
 6   value-units      5000 non-null   str           
 7   hour             5000 non-null   int64         
 8   dayofweek        5000 non-null   int64         
 9   month            5000 non-null   int64         
 10  year             5000 non-null   int64         
dtypes: datetime64[us](1), int64(5), str(5)
memory usage: 709.4 KB


## 1.3 Descriptive statistics

Note the negative `load_mw` minimum — this belongs to the **Total Interchange (TI)** type (exports), not a data error.

In [20]:
df.describe()

,timestamp,load_mw,hour,dayofweek,month,year
count,5000,5000.000000,5000.000000,5000.000000,5000.000000,5000.0
mean,2023-01-27 00:30:00,71042.770200,11.482400,2.960000,1.404800,2023.0
min,2023-01-01 00:00:00,-1346.000000,0.000000,0.000000,1.000000,2023.0
25%,2023-01-14 00:00:00,53404.750000,5.000000,1.000000,1.000000,2023.0
50%,2023-01-27 00:30:00,89728.000000,11.000000,3.000000,1.000000,2023.0
75%,2023-02-09 01:00:00,97075.750000,17.000000,5.000000,2.000000,2023.0
max,2023-02-22 01:00:00,122580.000000,23.000000,6.000000,2.000000,2023.0
std,NaN,39756.937518,6.931327,2.046079,0.490902,0.0


## 1.4 Missing value check

No nulls expected; confirm before proceeding to preprocessing.

In [21]:
missing = df.isnull().sum()
print(missing)
print(f'\nTotal nulls: {missing.sum()}')

timestamp          0
respondent         0
respondent-name    0
type               0
type-name          0
load_mw            0
value-units        0
hour               0
dayofweek          0
month              0
year               0
dtype: int64

Total nulls: 0


## 1.5 Data type breakdown

Confirm the four `type` codes (D, DF, NG, TI) are present and balanced.

In [22]:
print("Type distribution:")
print(df["type"].value_counts())
print(f'\nDate range: {df["timestamp"].min()} → {df["timestamp"].max()}')

Type distribution:
type
D     1250
DF    1250
NG    1250
TI    1250
Name: count, dtype: int64

Date range: 2023-01-01 00:00:00 → 2023-02-22 01:00:00
